In [ ]:
# -------------------------------------------------------------
# Libraries and Imports
# -------------------------------------------------------------

# Paths and File Handling
import abc
import subprocess
from pathlib import Path
from jinja2 import Template
from typing import Optional
import json
import uuid
from dataclasses import dataclass, field, asdict
from typing import Optional
import gmsh

# Numerical and Visualization Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pyvista as pv
import h5py

# Neural Network and Machine Learning Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from scipy.ndimage import label
from scipy.ndimage import distance_transform_edt
from scipy.spatial import cKDTree
from scipy.interpolate import griddata
from concurrent.futures import ThreadPoolExecutor

# Additional utilities
import os
import time
import random
import warnings
warnings.filterwarnings("ignore", message="An output with one or more elements was resized")

In [ ]:
# -------------------------------------------------------------
# Torch Device Configuration
# -------------------------------------------------------------
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")

In [ ]:
# -------------------------------------------------------------
# Geometry Generation Models
# -------------------------------------------------------------
class HeatExchangerCNN(nn.Module):
    """
    Maps a latent vector z -> occupancy grid of shape (1, grid_ny, grid_nx).

    The output is passed through sigmoid so values are in (0, 1).
    During training, keep the soft output for RL policy log-probs.
    For geometry generation, threshold at 'threshold' to get binary cells,
    then extract connected blobs as obstacle polygons.

    Architecture: fully-connected projection -> reshape -> series of 
    transposed conv upsampling blocks. The final spatial size is determined by grid_nx and grid_ny
    - the network is built to match these exactly.    
    """

    def __init__(
            self,
            latent_dim: int = 32,
            grid_nx: int = 20,
            grid_ny: int = 10,
            base_channels: int = 64,
    ):
        super().__init__()
        self.latent_dim = latent_dim
        self.grid_nx = grid_nx
        self.grid_ny = grid_ny
        self.base_channels = base_channels

        # Project z to a small spatial feature map, then upsample.
        # We start at (base_channels, 2, 2) and double the spatial dims each block.
        # Number of upsampling blocks needed:
        self._n_ups_x = int(np.ceil(np.log2(grid_nx))) - 1 # number of doublings needed
        self._n_ups_y = int(np.ceil(np.log2(grid_ny))) - 1
        n_blocks = max(self._n_ups_x, self._n_ups_y)

        # FC projection
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, base_channels * 4),
            nn.ReLU(),
            nn.Linear(base_channels * 4, base_channels * 2 * 2),
            nn.ReLU()
        )
        self.init_channels = base_channels
        self.init_h = 2
        self.init_w = 2

        # Upsampling blocks
        blocks = []
        in_ch = base_channels
        for i in range(n_blocks):
            out_ch = max(in_ch // 2, 8)
            blocks.append(nn.Sequential(
                nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
            ))
            in_ch = out_ch
        self.ups = nn.ModuleList(blocks)

        # Final 1x1 conv to singe-channel occupancy grid
        self.head = nn.Conv2d(in_ch, 1, kernel_size=1)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.ConvTranspose2d, nn.Conv2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """
        Parameters:
        z: Tensor of shape (B, latent_dim)
        
        Returns: 
        Tensor of shape (B, 1, grid_ny, grid_nx) occupancy grid with values in (0, 1)
        """
        B = z.shape[0]
        x = self.fc(z) # (B, base_channels * 2 * 2)
        x = x.view(B, self.init_channels, self.init_h, self.init_w) # (B, base_channels, 2, 2)

        for up in self.ups:
            x = up(x) # upsample
        x = self.head(x) # (B, 1, H, W)
        
        # Crop or interpolate to extract target size
        x = torch.nn.functional.interpolate(
            x, size=(self.grid_ny, self.grid_nx), mode='bilinear', align_corners=False
            )
        x = torch.sigmoid(x) # values in (0, 1)
        return x

In [ ]:
# -------------------------------------------------------------
# Geometry Filter and Differentiable Proxy Loss
# -------------------------------------------------------------
class GeometryFilter:
    """
    Validates an occupancy grid before committing to mesh generation.

    Checks
    ------
    1. Solid density: fraction of solid cells must be within [min_density, max_density].
    2. Flow path: at least one continuous fluid path from inlet (left) to outlet (right).
    3. No enclosed fluid: every fluid region must connect to both the inlet and outlet boundaries.
       Fluid pockets fully surrounded by solid are rejected.

    Parameters
    ----------
    min_density, max_density : float
        Solid fraction bounds.
    threshold : float
        Binarization threshold, should match HeatExchangerGenerator.
    """

    def __init__(
        self,
        min_density: float = 0.10,
        max_density: float = 0.75,
        threshold: float = 0.5,
    ):
        self.min_density = min_density
        self.max_density = max_density
        self.threshold = threshold

    def is_valid(self, occupancy: np.ndarray) -> tuple[bool, dict]:
        binary = (occupancy >= self.threshold)

        density_ok, density_val = self._check_density(binary)
        flow_ok                 = self._check_flow_path(binary)
        no_pockets              = self._check_no_enclosed_fluid(binary)

        report = {
            "solid_density":      round(float(density_val), 4),
            "density_ok":         density_ok,
            "has_flow_path":      flow_ok,
            "no_enclosed_fluid":  no_pockets,
            "valid":              density_ok and flow_ok and no_pockets,
        }
        return report["valid"], report

    def _check_density(self, binary: np.ndarray) -> tuple[bool, float]:
        density = binary.mean()
        return self.min_density <= density <= self.max_density, density

    def _check_flow_path(self, binary: np.ndarray) -> bool:
        fluid = ~binary
        if not fluid[:, 0].any() or not fluid[:, -1].any():
            return False
        labeled, n = label(fluid, structure=np.array([[0,1,0],[1,1,1],[0,1,0]]))
        for comp_id in range(1, n + 1):
            region = labeled == comp_id
            if region[:, 0].any() and region[:, -1].any():
                return True
        return False

    def _check_no_enclosed_fluid(self, binary: np.ndarray) -> bool:
        """
        Check that no fluid region is fully enclosed by solid.

        A fluid region is considered enclosed if it does not touch the inlet
        and outlet boundaries. Touching both boundaries means the fluid is reachable 
        from outside the heat exchanger, which is the physical requirement.
        """
        fluid = ~binary
        labeled, n = label(fluid, structure=np.array([[0,1,0],[1,1,1],[0,1,0]]))
        for comp_id in range(1, n + 1):
            region = labeled == comp_id
            touches_boundary = (
                region[:, 0].any()   and   # left column (inlet)
                region[:, -1].any()       # right column (outlet)
            )
            if not touches_boundary:
                return False
        return True
    
class FilterLoss(nn.Module):
    """
    Differentiable proxy for all three hard GeometryFilter checks. 
    Additionally, includes extra terms to encourage desirable properties like diversity, tortuosity, fragmentation, fine channel structure, sharp interfaces, and non-periodicity — all of which are common failure modes when training a CNN with only the basic filter checks as supervision.

    Parameters
    ----------
    min_density, max_density : float
        Target solid fraction bounds.
    density_weight : float
        Weight for the density loss term. Since density is a fundamental constraint, this should be set high enough to strongly penalize designs that are too sparse or too dense.
    flow_weight : float
        Weight for the flow path loss term. Since a clear flow path is critical, this should be set high to strongly penalize designs that block the inlet or outlet.
    pocket_weight : float
        Weight for the enclosed fluid loss term. Since enclosed fluid is a hard fail, this should be set high to strongly penalize any pockets.
    diversity_weight : float
        Weight for the diversity loss term, which encourages variation in the generated geometries across a batch of latent vectors.
    diversity_min_std : float
        Minimum standard deviation for the diversity loss term. If the standard deviation of the occupancy grids in a batch falls below this threshold, the diversity loss will increase to encourage more variation.
    tortuosity_weight : float
        Weight for the tortuosity loss term, which encourages more tortuous flow paths to break up straight channels that can cause periodic patterns.
    min_tortuosity : float
        Minimum tortuosity for the tortuosity loss term. If the tortuosity of the flow paths in a generated geometry falls below this threshold, the tortuosity loss will increase to encourage more winding paths.
    fragmentation_weight : float
        Weight for the fragmentation loss term, which discourages large blocks of solid or fluid and encourages more, smaller channels.
    channel_scale_weight : float
        Weight for the channel scale loss term, which encourages finer channel structures by penalizing deviations from a target channel width.
    target_channel_width : float
        Target channel width for the channel scale loss term. This pushes the generator toward producing features of a certain size, which can help avoid overly large solid blocks or wide fluid channels.
    interface_weight : float
        Weight for the interface loss term, which encourages sharper solid-fluid interfaces by penalizing intermediate occupancy values and promoting more binary-like outputs.
    periodicity_weight : float
        Weight for the periodicity loss term, which discourages periodic patterns in the generated geometries by penalizing repeating structures in the occupancy grid.
    """

    def __init__(
        self,
        min_density: float = 0.20,
        max_density: float = 0.80,
        density_weight: float = 3.0,            # increase to more strongly penalize densities outside the valid range
        flow_weight: float = 2.5,               # increase to strongly encourage a clear flow path
        pocket_weight: float = 7.5,             # enclosed fluid is a hard fail, increase to punish it more heavily
        diversity_weight: float = 3.0,          # increase to encourage more diversity in the batches
        diversity_min_std: float = 0.25,
        tortuosity_weight: float = 0.5,         # encourage tortuosity to break up straight channels that can cause periodic patterns
        min_tortuosity: float = 0.05, 
        fragmentation_weight: float = 5.0,      # increase to discourage large blocks of solid or fluid — encourages more, smaller channels
        channel_scale_weight: float = 4.0,      # increase to encourage fine structure
        target_channel_width: float = 0.10,     # pushes toward finer features
        interface_weight: float = 2.0,          # increase to encourage sharper solid-fluid interfaces
        periodicity_weight: float = 2.5,        # increase to discourage periodic patterns 
        sharpness_weight: float = 4.0,          # increase to encourage sharper transitions in the occupancy grid, reducing ambiguous values between solid and fluid
    ):
        super().__init__()
        self.min_density = min_density
        self.max_density = max_density
        self.density_weight = density_weight
        self.flow_weight = flow_weight
        self.pocket_weight = pocket_weight
        self.diversity_weight = diversity_weight
        self.diversity_min_std = diversity_min_std
        self.tortuosity_weight = tortuosity_weight
        self.min_tortuosity = min_tortuosity
        self.fragmentation_weight = fragmentation_weight
        self.channel_scale_weight = channel_scale_weight
        self.target_channel_width = target_channel_width
        self.interface_weight = interface_weight
        self.periodicity_weight = periodicity_weight
        self.sharpness_weight = sharpness_weight

    def forward(self, soft_grid):
        density_loss        = self._density_loss(soft_grid)
        flow_loss           = self._flow_loss(soft_grid)
        pocket_loss         = self._pocket_loss(soft_grid)
        diversity_loss      = self._diversity_loss(soft_grid, self.diversity_min_std)
        tortuosity_loss     = self._tortuosity_loss(soft_grid, self.min_tortuosity)
        fragmentation_loss  = self._fragmentation_loss(soft_grid)
        channel_scale_loss  = self._channel_scale_loss(soft_grid, self.target_channel_width)
        interface_loss      = self._interface_loss(soft_grid)
        periodicity_loss    = self._periodicity_loss(soft_grid)
        sharpness_loss      = self._sharpness_loss(soft_grid)

        total = (
            self.density_weight         * density_loss          +
            self.flow_weight            * flow_loss             +
            self.pocket_weight          * pocket_loss           +
            self.diversity_weight       * diversity_loss        +
            self.tortuosity_weight      * tortuosity_loss       +
            self.fragmentation_weight   * fragmentation_loss    +
            self.channel_scale_weight   * channel_scale_loss    +
            self.interface_weight       * interface_loss        +
            self.periodicity_weight     * periodicity_loss      +
            self.sharpness_weight       * sharpness_loss        
        )

        breakdown = {
            "density_loss":         density_loss.item(),
            "flow_loss":            flow_loss.item(),
            "pocket_loss":          pocket_loss.item(),
            "diversity_loss":       diversity_loss.item(),
            "tortuosity_loss":      tortuosity_loss.item(),
            "fragmentation_loss":   fragmentation_loss.item(),
            "channel_scale_loss":   channel_scale_loss.item(),
            "interface_loss":       interface_loss.item(),
            "periodicity_loss":     periodicity_loss.item(),
            "sharpness_loss":       sharpness_loss.item(),
            "total_loss":           total.item(),
        }

        return total, breakdown

    def _density_loss(self, soft_grid: torch.Tensor) -> torch.Tensor:
        """
        Penalizes designs that are too sparse or too dense, based on the average occupancy of the grid.
        """
        density_soft = soft_grid.mean(dim=[1, 2, 3])

        too_low_soft  = torch.relu(self.min_density - density_soft) ** 2
        # Asymmetric: penalize too-high more aggressively than too-low
        # since the current failure mode is density collapsing to 1.0
        too_high_soft = (torch.relu(density_soft - self.max_density) ** 2) * 3.0
        hinge_loss = (too_low_soft + too_high_soft).mean()

        target = (self.min_density + self.max_density) / 2.0
        pull_loss = (
            torch.relu(self.min_density - density_soft)
            * torch.abs(density_soft - target)
        ).mean()

        # Straight-through hard density
        binary = (soft_grid >= 0.5).float()
        binary_ste = soft_grid + (binary - soft_grid).detach()
        density_hard = binary_ste.mean(dim=[1, 2, 3])
        too_low_hard  = torch.relu(self.min_density - density_hard) ** 2
        too_high_hard = (torch.relu(density_hard - self.max_density) ** 2) * 3.0
        hard_loss = (too_low_hard + too_high_hard).mean()

        ambiguous = torch.relu(soft_grid - 0.3) * torch.relu(0.7 - soft_grid)

        # Edge-aware sharpening: ambiguous cells adjacent to decided cells
        # are penalized much more heavily than interior ambiguous cells.
        # A decided cell is one with value < 0.2 (fluid) or > 0.8 (solid).
        decided = ((soft_grid < 0.2) | (soft_grid > 0.8)).float()
        kernel = torch.ones(1, 1, 3, 3, device=soft_grid.device) / 9.0
        near_decided = torch.nn.functional.conv2d(decided, kernel, padding=1)
        # near_decided is high where there are decided neighbors — these are edge cells
        edge_weight = 1.0 + near_decided * 9.0   # up to 10x penalty at edges
        sharpening_loss = (ambiguous * edge_weight).mean() * 4.0

        return hinge_loss + pull_loss + hard_loss + sharpening_loss

    def _flow_loss(self, soft_grid: torch.Tensor) -> torch.Tensor:
        """
        Penalizes designs that lack a clear flow path from inlet to outlet.

        Switched to isotropic 3x3 kernel to remove horizontal stripe bias.
        """
        fluid = 1.0 - soft_grid

        inlet_blocking  = (1.0 - fluid[:, :, :, 0]).mean()
        outlet_blocking = (1.0 - fluid[:, :, :, -1]).mean()
        column_loss = inlet_blocking + outlet_blocking

        # Isotropic kernel — no directional bias
        kernel = torch.ones(1, 1, 3, 3, device=soft_grid.device) / 9.0
        neighbor_fluid = torch.nn.functional.conv2d(fluid, kernel, padding=1)
        connectivity_loss = torch.relu(fluid - neighbor_fluid).mean()

        return column_loss + connectivity_loss

    def _pocket_loss(self, soft_grid: torch.Tensor) -> torch.Tensor:
        """
        Differentiable flood-fill pocket detection, matching the updated
        GeometryFilter logic:

        - Reachability is seeded only from left (inlet) and right (outlet) columns.
        Top/bottom boundary contact does not count as reachable, matching the
        filter's requirement that fluid must connect inlet to outlet.
        - Two separate flood fills are run: one from the left, one from the right.
        A fluid cell is only considered valid if reachable from BOTH sides.
        This catches three failure modes:
            1. Enclosed pockets with no boundary contact
            2. Dead-end channels touching only the inlet or only the outlet
            3. Fluid regions touching only top/bottom walls
        """
        fluid = 1.0 - soft_grid
        B, _, H, W = fluid.shape

        # Fluid fraction guard — same as before
        fluid_fraction = fluid.mean(dim=[1, 2, 3])
        min_fluid = 1.0 - self.max_density
        fluid_shortage = torch.relu(min_fluid - fluid_fraction).mean()

        # Downsampling for large grids
        downsample_factor = 1
        if H * W > 50000:
            downsample_factor = 4
            fluid_ds = torch.nn.functional.avg_pool2d(
                fluid, kernel_size=downsample_factor, stride=downsample_factor,
            )
        else:
            fluid_ds = fluid

        _, _, Hd, Wd = fluid_ds.shape
        n_iters = (Hd + Wd) // 4

        def smooth_flood_fill(seed_col: int) -> torch.Tensor:
            """
            Flood fill using avg_pool2d instead of max_pool2d.

            avg_pool2d distributes gradient to all neighbors equally rather
            than only to the winner, giving the CNN a usable learning signal
            throughout the reachability map.

            The avg_pool with a scaling factor >1 approximates max behavior
            (high values dominate the average of their neighborhood) while
            keeping gradients alive. We clamp to [0,1] after each step to
            prevent runaway amplification.
            """
            reachable = torch.zeros_like(fluid_ds)
            reachable[:, :, :, seed_col] = fluid_ds[:, :, :, seed_col]

            # Reduce alpha from 4.0 to 2.0 — less amplification means fewer
            # cells hit the clamp ceiling, preserving more gradient paths.
            # Slower propagation speed is compensated by n_iters being based
            # on grid dimensions already.
            alpha = 2.0

            for _ in range(n_iters):
                expanded = torch.nn.functional.avg_pool2d(
                    reachable * alpha,
                    kernel_size=3,
                    stride=1,
                    padding=1,
                )
                # Use sigmoid instead of clamp — smooth saturation keeps gradients
                # alive near the ceiling rather than zeroing them out
                reachable = torch.sigmoid(expanded * 4.0 - 2.0) * fluid_ds
                reachable[:, :, :, seed_col] = fluid_ds[:, :, :, seed_col]

            return reachable

        reachable_from_left  = smooth_flood_fill(0)
        reachable_from_right = smooth_flood_fill(-1)

        if downsample_factor > 1:
            reachable_from_left = torch.nn.functional.interpolate(
                reachable_from_left,  size=(H, W), mode="bilinear", align_corners=False
            )
            reachable_from_right = torch.nn.functional.interpolate(
                reachable_from_right, size=(H, W), mode="bilinear", align_corners=False
            )

        reachable_both = torch.min(reachable_from_left, reachable_from_right)
        enclosed = fluid * torch.relu(fluid - reachable_both)

        return enclosed.mean() + fluid_shortage
    
    def _diversity_loss(
        self,
        soft_grid: torch.Tensor,
        min_std: float = 0.15,
    ) -> torch.Tensor:
        """
        Penalizes mode collapse by encouraging two kinds of diversity:

        1. Spatial variance within each sample — the occupancy values should
        vary across the grid, not be uniform. A collapsed CNN produces
        nearly identical values everywhere (e.g. all ~0.5), giving near-zero
        spatial std. We penalize when per-sample std drops below min_std.

        2. Inter-sample variance across the batch — different latent vectors
        should produce different grids. If all samples in a batch look the
        same, the mean of the batch looks like any individual sample, so the
        std across samples is near zero. We penalize that too.

        Parameters
        ----------
        soft_grid : Tensor of shape (B, 1, grid_ny, grid_nx)
        min_std : float
            Minimum acceptable std. Values below this are penalized.
            0.15 is a reasonable starting point — pure noise has std ~0.29,
            a fully collapsed output has std ~0.0.
        """
        # 1. Spatial variance: std across spatial dims for each sample
        spatial_std = soft_grid.std(dim=[2, 3])          # (B, 1)
        spatial_loss = torch.relu(min_std - spatial_std).mean()

        # 2. Inter-sample variance: std across batch dim at each spatial location
        if soft_grid.shape[0] > 1:
            batch_std = soft_grid.std(dim=0)             # (1, grid_ny, grid_nx)
            batch_loss = torch.relu(min_std - batch_std).mean()
        else:
            batch_loss = torch.tensor(0.0, device=soft_grid.device)

        return spatial_loss + batch_loss

    def _tortuosity_loss(
        self,
        soft_grid: torch.Tensor,
        min_tortuosity: float = 0.3,
    ) -> torch.Tensor:
        """
        Penalizes designs where fluid flows too directly from inlet to outlet.

        Tortuosity is approximated by measuring vertical variation in the
        fluid distribution across x-slices. A straight horizontal channel
        has the same fluid pattern at every x position (low variation).
        A snaking channel shifts up and down across x positions (high variation).

        Specifically: for each x column, compute the center of mass of the
        fluid cells in y. A path that snakes will have a y center-of-mass
        that varies significantly across x. We penalize when that variation
        is below min_tortuosity.

        Parameters
        ----------
        min_tortuosity : float
            Minimum acceptable std of the fluid y-centroid across x columns.
            Normalized to [0, 1] by grid height. 0.3 means the centroid
            should shift by at least 30% of the domain height across the length.
        """
        fluid = 1.0 - soft_grid  # (B, 1, grid_ny, grid_nx)
        B, _, H, W = fluid.shape

        # y coordinate for each row, normalized to [0, 1]
        y_coords = torch.linspace(0, 1, H, device=soft_grid.device)
        y_coords = y_coords.view(1, 1, H, 1).expand(B, 1, H, W)

        # Fluid-weighted y centroid at each x column: (B, 1, W)
        fluid_sum = fluid.sum(dim=2) + 1e-6          # avoid div by zero
        y_centroid = (fluid * y_coords).sum(dim=2) / fluid_sum  # (B, 1, W)

        # Std of the centroid across x — high for snaking, low for straight
        tortuosity = y_centroid.std(dim=2)           # (B, 1)

        # Penalize when tortuosity is below the minimum
        tortuosity_loss = torch.relu(min_tortuosity - tortuosity).mean()
        return tortuosity_loss
    
    
    def _channel_scale_loss(
        self,
        soft_grid: torch.Tensor,
        target_channel_width: float = 0.05,
    ) -> torch.Tensor:
        # Move to CPU for FFT to avoid MPS buffer resize warning,
        # then move result back to original device for the rest of the loss
        device = soft_grid.device
        solid = soft_grid.squeeze(1).cpu()
        B, H, W = solid.shape

        fft = torch.fft.rfft2(solid)
        power = fft.abs() ** 2

        total_power = power.sum(dim=[1, 2], keepdim=True) + 1e-8
        power_norm = power / total_power

        power_x = power_norm.mean(dim=1)   # (B, W//2+1)
        power_y = power_norm.mean(dim=2)   # (B, H)

        target_freq_x = max(1, int(W * target_channel_width))
        target_freq_y = max(1, int(H * target_channel_width))

        high_freq_x = power_x[:, target_freq_x:].sum(dim=1)
        high_freq_y = power_y[:, target_freq_y:].sum(dim=1)
        aspect_loss = torch.relu(high_freq_x - high_freq_y).mean()

        # Tighter band boundaries so the mid-band isn't trivially satisfied.
        # Previous: low_end=5, high_start=40 for H=100 — mid-band is 35 bins wide.
        # New: symmetric around target frequency so the band reward is meaningful.
        target_bin = max(1, int(H * target_channel_width))
        low_end    = max(1, target_bin // 2)
        high_start = min(H // 2, target_bin * 3)

        low_band_y  = power_y[:, 1:low_end].sum(dim=1)
        mid_band_y  = power_y[:, low_end:high_start].sum(dim=1)
        high_band_y = power_y[:, high_start:].sum(dim=1)

        mid_dominance_loss = (
            torch.relu(low_band_y  - mid_band_y) +
            torch.relu(high_band_y - mid_band_y)
        ).mean()

        # Additional term: penalize DC dominance directly.
        # DC component (index 0) represents the mean value — high DC power means
        # the grid is mostly one value (nearly all solid or all fluid).
        # Excluding DC from ac_power in periodicity_loss isn't enough since
        # channel_scale_loss needs to handle it separately.
        dc_power_y = power_y[:, 0]                    # (B,)
        ac_power_y = power_y[:, 1:].sum(dim=1)        # (B,)
        dc_ratio = dc_power_y / (ac_power_y + 1e-8)
        dc_ratio_clamped = torch.clamp(dc_ratio, 0.0, 20.0)  # prevent extreme values from destabilizing training
        dc_dominance_loss = torch.relu(dc_ratio_clamped - 2.0).mean()  # penalize DC > 2x AC

        return (aspect_loss + mid_dominance_loss + dc_dominance_loss).to(device)
    
    def _fragmentation_loss(self, soft_grid: torch.Tensor) -> torch.Tensor:
        """
        Combined solid and fluid fragmentation loss.

        Replaces the separate _solid_fragmentation_loss and _fluid_fragmentation_loss.
        Both had the same structure but applied to solid and fluid grids respectively,
        so combining them removes the weight imbalance between the two and makes
        tuning simpler — one weight controls the solid/fluid structural balance together.

        Penalizes:
        - Columns where solid blocks the full height (vertical_blockage)
        - Rows where fluid spans the full width (horizontal_openness)
        - Unimodal solid marginal distributions (one large solid blob)
        - Unimodal fluid marginal distributions (one large open space)

        The cross-symmetry between solid and fluid terms means neither phase
        can dominate spatially, which naturally pushes toward interleaved structure.
        """
        solid = soft_grid                   # (B, 1, H, W)
        fluid = 1.0 - soft_grid
        B, _, H, W = solid.shape

        # --- Solid terms ---
        # Penalize columns that are solid all the way through vertically
        col_min_solid = solid.min(dim=2).values           # (B, 1, W)
        vertical_blockage = col_min_solid.mean()

        solid_flat = solid.squeeze(1)                     # (B, H, W)
        solid_y_marginal = solid_flat.mean(dim=2)         # (B, H)
        solid_x_marginal = solid_flat.mean(dim=1)         # (B, W)
        solid_y_var = solid_y_marginal.var(dim=1)         # (B,)
        solid_x_var = solid_x_marginal.var(dim=1)         # (B,)

        min_var = 0.02
        solid_compactness = (
            torch.relu(min_var - solid_y_var) +
            torch.relu(min_var - solid_x_var)
        ).mean()

        # --- Fluid terms ---
        # Penalize rows that are fluid all the way through horizontally
        row_min_fluid = fluid.min(dim=3).values           # (B, 1, H)
        horizontal_openness = row_min_fluid.mean()

        fluid_flat = fluid.squeeze(1)                     # (B, H, W)
        fluid_y_marginal = fluid_flat.mean(dim=2)         # (B, H)
        fluid_x_marginal = fluid_flat.mean(dim=1)         # (B, W)
        fluid_y_var = fluid_y_marginal.var(dim=1)         # (B,)
        fluid_x_var = fluid_x_marginal.var(dim=1)         # (B,)

        fluid_compactness = (
            torch.relu(min_var - fluid_y_var) +
            torch.relu(min_var - fluid_x_var)
        ).mean()

        # --- Balance term ---
        # Penalize when solid and fluid span penalties are asymmetric.
        # If vertical_blockage >> horizontal_openness the design has tall solid blobs.
        # If horizontal_openness >> vertical_blockage it has wide fluid corridors.
        # We want them roughly equal, which corresponds to isotropic interleaving.
        balance_loss = (vertical_blockage - horizontal_openness).abs()

        return (
            vertical_blockage  +
            horizontal_openness +
            solid_compactness  +
            fluid_compactness  +
            balance_loss
        )

    def _interface_loss(self, soft_grid: torch.Tensor) -> torch.Tensor:
        """
        Rewards high solid-fluid interface density throughout the domain.

        A design with many snaking channels through a solid matrix has a very
        high perimeter-to-area ratio — lots of solid-fluid boundaries. A design
        with a few large blobs or a few wide channels has low interface density.
        This term directly rewards the kind of fine, interleaved structure you want.

        Computed using Sobel-like finite differences on the soft grid — regions
        where the gradient magnitude is high are solid-fluid boundaries.
        We reward high mean gradient magnitude across the domain.

        Additionally penalizes when interface is spatially concentrated 
        (e.g. all boundaries are in one region) rather than spread evenly.
        """
        solid = soft_grid

        grad_x = solid[:, :, :, 1:] - solid[:, :, :, :-1]
        grad_y = solid[:, :, 1:, :] - solid[:, :, :-1, :]

        grad_x_pad = torch.nn.functional.pad(grad_x.abs(), (0, 1))
        grad_y_pad = torch.nn.functional.pad(grad_y.abs(), (0, 0, 0, 1))
        grad_mag = (grad_x_pad + grad_y_pad) / 2.0

        # Replace fixed threshold with a relative target.
        # A perfectly binary grid with 50% solid has mean gradient ~0.5 * 2/cell_size.
        # We target 30% of that theoretical maximum as a minimum, which is
        # high enough to require actual sharp transitions rather than smooth gradients.
        theoretical_max = 0.5   # max possible mean gradient for a binary 50/50 grid
        target_interface = theoretical_max * 0.30
        mean_interface = grad_mag.mean()
        interface_density_loss = torch.relu(target_interface - mean_interface)

        # Spatial concentration — unchanged
        B, _, H, W = grad_mag.shape
        h_third, w_third = H // 3, W // 3
        region_means = []
        for i in range(3):
            for j in range(3):
                region = grad_mag[
                    :, :,
                    i*h_third:(i+1)*h_third,
                    j*w_third:(j+1)*w_third,
                ]
                region_means.append(region.mean())

        region_means = torch.stack(region_means)
        concentration_loss = region_means.var()

        # Additional term: penalize smooth gradients explicitly.
        # A soft grid with values clustering near 0.5 has low gradient variance —
        # all gradients are small. A sharp binary grid has high gradient variance
        # (most gradients are ~0, transitions are ~1). Penalize low gradient variance.
        grad_var = grad_mag.var()
        sharpness_loss = torch.relu(0.02 - grad_var)

        return interface_density_loss + concentration_loss + sharpness_loss
    
    def _periodicity_loss(self, soft_grid: torch.Tensor) -> torch.Tensor:
        device = soft_grid.device
        solid = soft_grid.squeeze(1).cpu()      # (B, H, W) on CPU

        def axis_periodicity(tensor, dim):
            fft = torch.fft.rfft(tensor, dim=dim)
            power = fft.abs() ** 2
            other_dim = 2 if dim == 1 else 1
            mean_power = power.mean(dim=other_dim)
            ac_power = mean_power[:, 1:]
            ac_total = ac_power.sum(dim=1, keepdim=True) + 1e-6
            ac_norm = ac_power / ac_total
            peak_power = ac_norm.max(dim=1).values
            return torch.relu(peak_power - 0.20).mean()

        x_periodicity = axis_periodicity(solid, dim=2)
        y_periodicity = axis_periodicity(solid, dim=1)

        return (x_periodicity + y_periodicity).to(device)  # back to MPS
    
    def _sharpness_loss(self, soft_grid: torch.Tensor) -> torch.Tensor:
        """
        Penalizes wide gradient transitions at solid-fluid boundaries.

        A sharp binary boundary has a transition width of 1 cell — values
        jump from near-0 to near-1 in a single step. A blurry boundary has
        a wide ramp of intermediate values over many cells.

        We measure this by looking at gradient magnitude alongside ambiguity.
        At a sharp edge, gradient magnitude is high AND neighboring values
        are decided (near 0 or 1). At a blurry edge, gradient magnitude is
        moderate AND many neighbors are also ambiguous.

        Two components:
        1. Ambiguous neighbor penalty: penalize cells that are ambiguous AND
        have many ambiguous neighbors — these are interior fringe cells,
        not edge cells transitioning from one phase to another.
        2. Gradient-ambiguity mismatch: at a true edge, ambiguous cells should
        have high gradient magnitude. Penalize ambiguous cells with LOW
        gradient (flat intermediate values = fringe, not transition).
        """
        # Ambiguity map: high for values near 0.5, zero at 0 or 1
        ambiguous = torch.relu(soft_grid - 0.2) * torch.relu(0.8 - soft_grid)
        ambiguous_norm = ambiguous / (ambiguous.max() + 1e-8)

        # 1. Ambiguous neighbor penalty
        kernel = torch.ones(1, 1, 3, 3, device=soft_grid.device) / 9.0
        neighbor_ambiguity = torch.nn.functional.conv2d(
            ambiguous_norm, kernel, padding=1
        )
        # High neighbor ambiguity = surrounded by fringe = interior fringe cell
        fringe_penalty = (ambiguous_norm * neighbor_ambiguity).mean()

        # 2. Gradient-ambiguity mismatch
        grad_x = (soft_grid[:, :, :, 1:] - soft_grid[:, :, :, :-1]).abs()
        grad_y = (soft_grid[:, :, 1:, :] - soft_grid[:, :, :-1, :]).abs()
        grad_x_pad = torch.nn.functional.pad(grad_x, (0, 1))
        grad_y_pad = torch.nn.functional.pad(grad_y, (0, 0, 0, 1))
        grad_mag = (grad_x_pad + grad_y_pad) / 2.0

        # Ambiguous cells should have high gradient (they're at a transition).
        # Penalize ambiguous cells with low gradient — these are flat fringe cells.
        flat_fringe = ambiguous_norm * torch.relu(0.3 - grad_mag)
        flat_fringe_penalty = flat_fringe.mean()

        return fringe_penalty + flat_fringe_penalty

In [ ]:
# ------------------------------------------
# GeometryConfig dataclass for serialization
# ------------------------------------------
@dataclass
class GeometryConfig:
    """
    Serializable snapshot of a heat exchanger geometry.
    
    Produced by the Generator and consumed by both the MeshGenerator (to build the .msh file)
    and the Optimizer (to map simulation results back onto the original grid).
    
    Fields:
    -------
    grid_nx, grid_ny: int
        Number of grid points in the x and y directions for the reference grid.
    domain_length, domain_height: float
        Physical dimensions of the rectangular fluid domain.
    occupancy_grid : list[list[float]]
        2D list representing the occupancy of the domain, where 0 indicates solid and 1 indicates fluid.Soft (0-1) CNN output before thresholding. Shape: (grid_ny, grid_nx).
        Stored so the CNN state can be reproduced exactly for the same geometry, and so the optimizer can use the same grid for mapping results back to the original geometry.
    obstacle_polygons : list[list[tuple[float, float]]]
        List of polygons representing the obstructions in the geometry. Each polygon is a list of (x, y) coordinates of its vertices.
    config_id : str
        UUID assigned at creation - ties a GeometryConfig to its .msh file and simulation results for reproducibility and traceability.
    """

    grid_nx: int
    grid_ny: int
    domain_length: float
    domain_height: float
    threshold: float
    occupancy_grid: list # shape (grid_ny, grid_nx), values between 0 and 1
    obstacle_polygons: list # list of polygons, where each polygon is a list of (x, y) vertex coordinates
    config_id: str = field(default_factory=lambda: str(uuid.uuid4()))

    # Serialization
    def save(self, path: str | Path) -> None:
        """Save to a JSON file. Mesh files use the same config_id stem."""
        filename = path/f"hx_{self.config_id}.json"
        with open(filename, "w") as f:
            json.dump(asdict(self), f, indent=2)
    
    @classmethod
    def load(cls, path: str | Path) -> "GeometryConfig":
        with open(path) as f:
            data = json.load(f)
        return cls(**data)
    
    @property
    def mesh_filename(self) -> str:
        """Canonical filename derived from config_id."""
        return f"hx_{self.config_id}.msh"
    
    def grid_cell_size(self) -> tuple[float, float]:
        """Physical size of each grid cell (dx, dy)."""
        return (self.domain_length / self.grid_nx, self.domain_height / self.grid_ny)
    
    def cell_center(self, ix: int, iy: int) -> tuple[float, float]:
        """
        Physical (x, y) coordinates of the center of grid cell (ix, iy).
        ix in [0, grid_nx), iy in [0, grid_ny).
        Origin is at the bottome left of the domain
        """
        dx, dy = self.grid_cell_size()
        x = (ix + 0.5) * dx
        y = (iy + 0.5) * dy
        return (x, y)

In [ ]:
# -------------------------------------------------------
# HeatExchanger dataclass for geometry, mesh, and results
# -------------------------------------------------------
class HeatExchanger:
    """
    Contains the geometry, mesh, and simulation results for a single heat exchanger design. This is the main data structure that the Generator produces and the Optimizer consumes.

    Parameters:
    id: str
        Unique identifier for this heat exchanger design, typically derived from the GeometryConfig's config_id.
    config_filename: Optional[str]
        Path to the GeometryConfig JSON file that describes the geometry of this heat exchanger. This file is used to generate the mesh and to map simulation results back to the original grid.
    mesh_filename: Optional[str]
        Path to the GMSH .msh file that contains the mesh for this heat exchanger design. This file is generated from the GeometryConfig and is used as input for the simulation.
    result_filename: Optional[str]
        Path to the simulation results file (e.g., Exodus .e file) for this heat exchanger design. This file is generated by running the simulation on the mesh and contains the results that will be analyzed and used for optimization.
    geometry_config: Optional[GeometryConfig]
        An instance of GeometryConfig that describes the geometry of this heat exchanger. This is typically loaded from the config_filename and is used for reference throughout the optimization process.
    obstruction_polygons: Optional[list]
        A list of polygons representing the obstructions in the geometry, extracted from the occupancy grid. Each polygon is a list of (x, y) coordinates of its vertices. This is derived from the GeometryConfig and is used for mesh generation and visualization.
    solved: bool
        A flag indicating whether the simulation for this heat exchanger design has been run and results are available. This is used to track the state of the optimization process.
    results: Optional[dict]
        A dictionary containing the analyzed results from the simulation, such as temperature distribution, flow patterns, and performance metrics. This is derived from the result_filename and is used for optimization and decision-making.
    """
    
    def __init__(
        self,
        id: str,
        mesh_params: dict,
        geometry_config: Optional[GeometryConfig] = None,
        mesh_scale: float = 1.0,
        config_dir: str | Path = "configs",
        mesh_dir: str | Path = "meshes",
        result_dir: str | Path = "results"
    ):
        self.id = id
        self.mesh_params = mesh_params
        self.mesh_scale = mesh_scale
        self.config_filename = None
        self.mesh_filename = None
        self.result_filename = None
        self.config_dir = Path(config_dir)
        self.mesh_dir = Path(mesh_dir)
        self.result_dir = Path(result_dir)
        self.set_filenames()
        
        # Load geometry config and extract obstruction polygons
        self.geometry_config = geometry_config
        self.obstruction_polygons = None
        if geometry_config is None:
            self.load_geometry_config()
        else:
            self.obstruction_polygons = self.geometry_config.obstacle_polygons

        # Check for existing mesh and results
        self.mesh_exists = self.check_for_mesh()
        
        # Simulation state
        self.results = None
        self.solved = self.check_for_existing_results()
    
    def set_filenames(self):
        """Set the mesh_filename and result_filename based on the heat exchanger ID."""
        self.config_filename = f"{self.config_dir}/hx_{self.id}.json"
        self.mesh_filename = f"{self.mesh_dir}/hx_{self.id}.msh"
        self.result_filename = f"{self.result_dir}/hx_{self.id}.e"

    def load_geometry_config(self):
        """Load the GeometryConfig from the config_filename."""
        self.geometry_config = GeometryConfig.load(self.config_filename)
        self.obstruction_polygons = self.geometry_config.obstacle_polygons

    def check_for_mesh(self) -> bool:
        """Check if the mesh file already exists for this heat exchanger design."""
        if os.path.exists(self.mesh_filename):
            return True
        else:
            self.generate_mesh()
        return os.path.exists(self.mesh_filename)

    def check_for_existing_results(self) -> bool:
        """Check if the result file already exists for this heat exchanger design."""
        if os.path.exists(self.result_filename):
            self.read_results()
        return os.path.exists(self.result_filename)
    
    def read_results(self):
        """Read the simulation results from the result_filename and store them in self.results."""
        # Placeholder for reading results - this would depend on the format of the results file
        self.results = None

    def create_custom_obstruction(self, points: list[tuple[float, float]]) -> int:
        """
        Create a planar surface in GMSH from an ordered list of (x, y) points.
        Returns the GMSH surface tag.
        """
        n_points = len(points)
        gmsh_points = []
        for coord in points:
            pt = gmsh.model.occ.addPoint(coord[0], coord[1], 0)
            gmsh_points.append(pt)
 
        lines = []
        for i in range(n_points):
            line = gmsh.model.occ.addLine(
                gmsh_points[i],
                gmsh_points[(i + 1) % n_points],
            )
            lines.append(line)
 
        cl = gmsh.model.occ.addCurveLoop(lines)
        surface = gmsh.model.occ.addPlaneSurface([cl])
        return surface

    def generate_mesh(self, verbose: bool = False) -> str:
        """
        Returns the path to the written .msh file.
        Produces a fully structured quad mesh aligned to the dx/dy grid
        """
        # Initialize gmsh
        gmsh.initialize()
        gmsh.option.setNumber("General.Terminal", 1 if verbose else 0)
        gmsh.model.add(f"hx_{self.id}")
        if not verbose:
            print(f"Generating mesh for heat exchanger {self.id}")
            start = time.time()
        
        # Extract grid geometry from geometry_config and build base domain
        Lx = self.geometry_config.domain_length
        Ly = self.geometry_config.domain_height
        nx = self.geometry_config.grid_nx
        ny = self.geometry_config.grid_ny
        dx = Lx / (nx * self.mesh_scale)
        dy = Ly / (ny * self.mesh_scale)
        rect = gmsh.model.occ.addRectangle(
            0, 0, 0,
            Lx, Ly,
        )
        gmsh.model.occ.synchronize()

        # Create obstruction surfaces from polygons
        obstructions = []
        for polygon in self.obstruction_polygons:
            surface = self.create_custom_obstruction(polygon)
            obstructions.append(surface)
        gmsh.model.occ.synchronize()
 
        if obstructions:
            cut = gmsh.model.occ.cut(
                [(2, rect)],
                [(2, s) for s in obstructions],
                removeObject=True,
                removeTool=True,
            )
            gmsh.model.occ.synchronize()
            out_dim_tags, _ = cut
            fluid_surfaces = [tag for dim, tag in out_dim_tags if dim == 2]
        else:
            # No obstacles — entire rectangle is the fluid domain
            fluid_surfaces = [rect]
 
        if not fluid_surfaces:
            gmsh.finalize()
            raise RuntimeError(
                "Boolean cut produced no fluid domain. "
                "Check that obstacles don't fill the entire domain."
            )

        # Add vertical and horizontal lines to create a structured grid aligned with the occupancy grid
        v_lines = []
        for i in range(1, nx * self.mesh_scale):
            x = i * dx
            p1 = gmsh.model.occ.addPoint(x,0,0)
            p2 = gmsh.model.occ.addPoint(x,Ly,0)
            v_lines.append(gmsh.model.occ.addLine(p1, p2))
        
        h_lines = []
        for j in range(1, ny * self.mesh_scale):
            y = j * dy
            p1 = gmsh.model.occ.addPoint(0,y,0)
            p2 = gmsh.model.occ.addPoint(Lx,y,0)
            h_lines.append(gmsh.model.occ.addLine(p1, p2))
        
        gmsh.model.occ.synchronize()

        # Split the surface at every grid line to create a structured mesh
        all_grid_curves = [(1, l) for l in v_lines + h_lines]
        frag, _ = gmsh.model.occ.fragment(
            [(2, s) for s in fluid_surfaces],
            all_grid_curves
        )
        gmsh.model.occ.synchronize()

        fluid_surfaces = [tag for dim, tag in frag if dim == 2]
        if not fluid_surfaces:
            gmsh.finalize()
            raise RuntimeError(
                "Fragmentation produced no fluid domain. "
                "Check that grid lines are correctly defined."
            )
 
        gmsh.model.addPhysicalGroup(2, fluid_surfaces, name="Fluid")

        # Force transfinite meshing to get a structured quad mesh aligned to the grid
        all_curves = gmsh.model.getEntities(1)
        for dim, tag in all_curves:
            # get center of mass to measure curve length indirectly
            xmin, ymin, zmin, xmax, ymax, zmax = gmsh.model.occ.getBoundingBox(dim, tag)
            length = max(abs(xmax - xmin), abs(ymax - ymin))

            # 1 node interval per dx or dy: nodes = 2 (endpoints only)
            if abs(xmax - xmin) > abs(ymax - ymin):
                n_cells = max(1, round(abs(xmax - xmin) / dx))
            else:
                n_cells = max(1, round(abs(ymax - ymin) / dy))
            gmsh.model.mesh.setTransfiniteCurve(tag, n_cells + 1)

        for tag in fluid_surfaces:
            try:
                gmsh.model.mesh.setTransfiniteSurface(tag)
                gmsh.model.mesh.setRecombine(2, tag) # recombine triangles into quads
            except Exception as e:
                gmsh.model.mesh.setRecombine(2, tag) # if transfinite fails (e.g. due to geometry issues), at least try to recombine into quads
                
        # Boundary classification
        gmsh.model.occ.synchronize()
        boundaries = gmsh.model.getBoundary(
            [(2, s) for s in fluid_surfaces], oriented=False
        )

        # Got rid of top and bottom classifications for now
        wall_curves, inlet, outlet = [], [], []
        seen = set()
        for dim, tag in boundaries:
            if tag in seen:
                continue
            seen.add(tag)
            xmin, ymin, _, xmax, ymax, _ = gmsh.model.occ.getBoundingBox(dim, tag)
            if abs(xmin) < 1e-6:
                inlet.append(tag)
            elif abs(xmax - Lx) < 1e-6:
                outlet.append(tag)
            else:
                wall_curves.append(tag)
 
        if wall_curves:
            gmsh.model.addPhysicalGroup(1, wall_curves, name="Wall")
        if inlet:
            gmsh.model.addPhysicalGroup(1, inlet, name="Inlet")
        if outlet:
            gmsh.model.addPhysicalGroup(1, outlet, name="Outlet")

        # Set mesh params and generate mesh
        p = self.mesh_params
        gmsh.option.setNumber("Mesh.CharacteristicLengthMin",   p["mesh_min"])
        gmsh.option.setNumber("Mesh.CharacteristicLengthMax",   p["mesh_max"])
        gmsh.option.setNumber("Mesh.Algorithm",                 p["mesh_algorithm"])
        gmsh.option.setNumber("Mesh.RecombineAll",              p["mesh_recombine"])
        gmsh.option.setNumber("Mesh.RecombinationAlgorithm",    p["recombination_algorithm"])
        gmsh.option.setNumber("Mesh.ElementOrder",              p["mesh_element_order"])
        gmsh.option.setNumber("Mesh.SecondOrderLinear",         p["mesh_second_order_linear"])
 
        gmsh.model.mesh.generate(2)
        gmsh.write(self.mesh_filename)
        gmsh.finalize()

        if not verbose:
            print(f"Finished generating mesh for heat exchanger {self.id} in {time.time() - start:.2f} seconds")

        return self.mesh_filename

In [ ]:
# -------------------------------------------------------------
# Primary Geometry Generator Class
# -------------------------------------------------------------
class HeatExchangerGenerator:
    """
    Wraps the CNN and all geometry post-processing.

    Used by the Optimizer to:
    - Sample a latent vector z (or receive one from the RL policy).
    - Produce a GeometryConfig (occupancy grid + polygon list + metadata)
    - Pass the config to MeshGenerator to write a .msh file

    Parameters:
    grid_nx, grid_ny: int
        Resolution of the occupancy grid used for geometry representation and CNN output.
    domain_length, domain_height: float
        Physical dimensions of the heat exchanger domain (in meters or consistent units).
    latent_dim: int
        Dimensions of the CNN latent space.
    threshold: float
        Threshold for converting the CNN's soft occupancy output into binary solid/fluid classification.
    device: str
        PyTorch device to run the CNN on ('cuda', 'mps', or 'cpu').
    mesh_params: dict | None
        Optional GMSH meshing parameters to forward to MeshGenerator. If None, defaults will be used.
    mesh_scale: float
        Scaling factor between geometry generation grid and CFD mesh. A mesh_scale > 1.0 means the mesh will be finer than the geometry grid. Works best when using integer scaling factors (e.g., 2.0 for half the cell size) to maintain alignment between geometry and mesh.
    """

    def __init__(
        self,
        grid_nx: int = 20,
        grid_ny: int = 10,
        domain_length: float = 1.0,
        domain_height: float = 0.5,
        latent_dim: int = 32,
        threshold: float = 0.5,
        device: str = "cpu",
        min_density: float = 0.25,
        max_density: float = 0.70,
        config_directory: str | Path = "configs",
        mesh_directory: str | Path = "meshes",
        result_directory: str | Path = "results",
        mesh_params: Optional[dict] = None,
        mesh_scale: float = 1.0,
    ):
        self.grid_nx = grid_nx
        self.grid_ny = grid_ny
        self.domain_length = domain_length
        self.domain_height = domain_height
        self.dx = domain_length / grid_nx
        self.dy = domain_height / grid_ny
        self.config_directory = Path(config_directory)
        self.mesh_directory = Path(mesh_directory)
        self.result_directory = Path(result_directory)
        self.history = None

        self.mesh_scale = mesh_scale
        self.mesh_params = mesh_params or {
            "mesh_algorithm": 5, # Delaunay
            "mesh_recombine": 1,
            "recombination_algorithm": 2, # Blossom - best quad
            "mesh_element_order": 2,
            "mesh_second_order_linear": 1,
        }
        self.mesh_params["mesh_min"] = min(self.dx / self.mesh_scale, self.dy / self.mesh_scale)
        self.mesh_params["mesh_max"] = max(self.dx / self.mesh_scale, self.dy / self.mesh_scale)
        

        self.latent_dim = latent_dim
        self.threshold = threshold
        self.device = torch.device(device)
        
        # Using the RandomNoiseGenerator as a placeholder for the CNN during development.
        self.min_density = min_density
        self.max_density = max_density

        self.cnn = HeatExchangerCNN(
            latent_dim=latent_dim,
            grid_nx=grid_nx,
            grid_ny=grid_ny,
            base_channels=64
        ).to(self.device)
        
        self.filter = GeometryFilter(
            min_density=self.min_density,
            max_density=self.max_density,
            threshold=threshold,
        )

    # Main Generation Function
    def generate(
        self,
        z: Optional[torch.Tensor] = None,
        config_id: Optional[str] = None,
        print_polygons: bool = False,
        diagnostic: bool = True,
    ) -> tuple["GeometryConfig", torch.Tensor, "HeatExchanger"] | tuple[None, None, None]:
        """
        Generate a heat exchanger geometry from a latent vector z.

        Returns (None, None) if the generated occupancy grid fails the
        GeometryFilter checks. The caller (generate_batch) will simply
        resample in that case.

        Parameters
        ----------
        z : Optional[torch.Tensor]
            Latent vector of shape (1, latent_dim). Sampled from N(0,1) if None.
        config_id : Optional[str]
            UUID string to assign. Auto-generated if None.
        print_polygons : bool
            If True, print the vertices of the generated polygons to the console for debugging.

        Returns
        -------
        (GeometryConfig, soft_grid_tensor) on success, (None, None) on filter failure.
        """
        if z is None:
            z = torch.randn(1, self.latent_dim, device=self.device)

        self.cnn.eval()
        with torch.no_grad():
            soft_grid = self.cnn(z)  # (1, 1, grid_ny, grid_nx)

        occupancy_np = soft_grid.squeeze().cpu().numpy()  # (grid_ny, grid_nx)

        # --- viability filter ---
        valid, report = self.filter.is_valid(occupancy_np)
        if not valid:
            return None, None, None

        obstacle_polygons = self.occupancy_to_polygons(
            occupancy=occupancy_np,
            domain_length=self.domain_length,
            domain_height=self.domain_height,
            threshold=self.threshold,
            min_cells=1,
            print_polygons=print_polygons,
        )

        config = GeometryConfig(
            grid_nx=self.grid_nx,
            grid_ny=self.grid_ny,
            domain_length=self.domain_length,
            domain_height=self.domain_height,
            threshold=self.threshold,
            occupancy_grid=occupancy_np.tolist(),
            obstacle_polygons=obstacle_polygons,
            config_id=config_id or str(uuid.uuid4()),
        )
        config.save(self.config_directory)

        hx = HeatExchanger(
            id=config.config_id,
            mesh_params=self.mesh_params,
            geometry_config=config,
            mesh_scale=self.mesh_scale,
            config_dir=self.config_directory,
            mesh_dir=self.mesh_directory,
            result_dir=self.result_directory
        )

        if diagnostic:
            print(f"Generated heat exchanger {config.config_id} with {len(obstacle_polygons)} obstacles.")
            self.plot_mesh_diagnostic(hx)

        return config, soft_grid, hx
    
    # Additional methods for training the CNN, saving/loading model weights, etc.
    def save_weights(self, path: str | Path) -> None:
        torch.save(self.cnn.state_dict(), path)

    def load_weights(self, path: str | Path) -> None:
        self.cnn.load_state_dict(torch.load(path, map_location=self.device))
    
    def parameters(self):
        # Expose CNN parameters for optimization/training
        return self.cnn.parameters()
    
    def pretrain_on_filter(
        self,
        n_steps: int = 2000,
        batch_size: int = 16,
        lr: float = 1e-3,
        log_every: int = 100,
        num_workers: int = 4,
    ) -> list[dict]:
        """
        Stage 1 training with full GPU acceleration. Teaches the generator to produce outputs that satisfy the GeometryFilter constraints, which is a prerequisite for successful optimization in stage 2.
        
        Parameters
        ----------
        num_workers : int
            Number of threads for parallel hard filter evaluation. Set to
            min(batch_size, cpu_count) for best performance. These run
            concurrently with GPU training so they're effectively free.
        """
        # Device setup
        device = self.device
        use_amp = device.type == "cuda"  # autocast stable on CUDA; MPS support is partial
        print(f"Training on device: {device} | AMP: {use_amp}")

        # Move criterion to the same device so all loss tensors stay on GPU
        criterion = FilterLoss(
            min_density=self.min_density,
            max_density=self.max_density,
        ).to(device)

        optimizer = torch.optim.Adam(self.cnn.parameters(), lr=lr)

        # GradScaler only used with CUDA AMP
        scaler = torch.amp.GradScaler(enabled=use_amp)

        # Scheduler: reduce LR if loss plateaus
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=5,      # in units of log_every steps
            min_lr=1e-5,
        )

        history = []
        self.cnn.train()
        executor = ThreadPoolExecutor(max_workers=num_workers)

        def _check_single(grid_np):
            """Run hard filter on one numpy grid. Called in thread pool."""
            valid, _ = self.filter.is_valid(grid_np)
            return valid

        def _submit_filter_checks(soft_grid_detached):
            """
            Submit batch filter checks to thread pool.
            Returns a list of Future objects — call .result() to get values.
            Grids are transferred to CPU once here, not per-sample.
            """
            grids_np = soft_grid_detached.cpu().numpy()   # single transfer
            return [
                executor.submit(_check_single, grids_np[i, 0])
                for i in range(grids_np.shape[0])
            ]

        pending_futures = None   # filter check futures from previous log step
        pending_step = None      # which step those futures belong to

        # Training loop
        for step in range(n_steps):
            z = torch.randn(batch_size, self.latent_dim, device=device)

            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                soft_grid = self.cnn(z)                    # (B, 1, H, W) on GPU
                loss, breakdown = criterion(soft_grid)

            optimizer.zero_grad(set_to_none=True)          # faster than zero_grad()
            scaler.scale(loss).backward()

            # Gradient clipping — important at 2M cell resolution where
            # spatial loss terms produce large raw gradients
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(self.cnn.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()

            # Logging and filter evaluation every log_every steps. Filter checks run in parallel on CPU while GPU continues training.
            if step % log_every == 0:
                # Collect previous parallel filter results if available
                if pending_futures is not None:
                    pass_count = sum(f.result() for f in pending_futures)
                    pass_rate = pass_count / batch_size
                else:
                    pass_rate = float("nan")   # not yet available at step 0

                # Submit filter checks for THIS step's output (runs in background)
                pending_futures = _submit_filter_checks(soft_grid.detach())
                pending_step = step

                breakdown["filter_pass_rate"] = pass_rate
                breakdown["step"] = step
                breakdown["lr"] = optimizer.param_groups[0]["lr"]
                history.append(breakdown)

                print(
                    f"Step {step:4d} | "
                    f"pass {pass_rate:.2f} | "
                    f"loss {breakdown['total_loss']:.4f} | "
                    f"rho {breakdown['density_loss']:.4f} | "
                    f"flow {breakdown['flow_loss']:.4f} | "
                    f"pocket {breakdown['pocket_loss']:.4f} | "
                    f"div {breakdown['diversity_loss']:.4f} | "
                    f"tort {breakdown['tortuosity_loss']:.4f} | "
                    f"frag {breakdown['fragmentation_loss']:.4f} | "
                    f"chan {breakdown['channel_scale_loss']:.4f} | "
                    f"int {breakdown['interface_loss']:.4f} | "
                    f"per {breakdown['periodicity_loss']:.4f} | "
                    f"sharp {breakdown['sharpness_loss']:.4f} | "
                    f"lr {breakdown['lr']:.2e}"
                )

                scheduler.step(breakdown["total_loss"])

        # Final pass rate evaluation on 50 samples to verify final model performance after training
        print("\nEvaluating final pass rate on 50 samples...")
        self.cnn.eval()
        final_futures = []
        with torch.no_grad():
            for _ in range(50):
                z = torch.randn(1, self.latent_dim, device=device)
                with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                    grid = self.cnn(z)
                final_futures.append(executor.submit(_check_single, grid[0, 0].cpu().numpy()))

        final_pass_rate = sum(f.result() for f in final_futures) / 50
        print(f"Final pass rate: {final_pass_rate:.2f}")

        executor.shutdown(wait=True)
        self.cnn.train()

        self.history = history
        return history

    # Geometry post-processing methods
    def occupancy_to_polygons(
        self,
        occupancy: np.ndarray,
        domain_length: float,
        domain_height: float,
        threshold: float = 0.5,
        min_cells: int = 1,
        method: str = "boundary",
        print_polygons: bool = False,
    ) -> list[list[tuple[float, float]]]:
        """
        Convert a thresholded occupancy grid into obstacle polygons for GMSH.
        Two methods are available, selectable via the `method` parameter:

        'boundary' (default)
        --------------------
        Traces the exact outer perimeter of each connected solid component
        as a single rectilinear (staircase) polygon. 

        'per_cell'
        ----------
        Creates one dx x dy rectangle per solid cell. 

        Parameters
        ----------
        occupancy : np.ndarray, shape (grid_ny, grid_nx)
            Soft CNN output, values in [0, 1].
        domain_length, domain_height : float
            Physical dimensions of the domain.
        threshold : float
            Binarisation threshold (default 0.5).
        min_cells : int
            Minimum number of solid cells for a component to be included.
        method : str
            'boundary' (default) or 'per_cell'.
        print_polygons : bool
            If True, print vertex coordinates of each polygon to stdout.

        Returns
        -------
        list of polygons. Each polygon is a list of (x, y) tuples in CCW
        order giving the physical coordinates of the polygon vertices.
        y is measured upward from the bottom of the domain (origin at
        bottom-left), consistent with GMSH and GeometryConfig.cell_center.
        """
        from scipy.ndimage import label as ndlabel

        grid_ny, grid_nx = occupancy.shape
        dx = domain_length / grid_nx
        dy = domain_height / grid_ny

        binary = (occupancy >= threshold).astype(int)
        labeled, n_components = ndlabel(
            binary, structure=np.array([[0,1,0],[1,1,1],[0,1,0]])
        )

        polygons = []
        for comp_id in range(1, n_components + 1):
            cells = np.argwhere(labeled == comp_id)   # (N, 2)  each row: (iy, ix)
            if len(cells) < min_cells:
                continue

            if method == "per_cell":
                for iy, ix in cells:
                    polygons.append(self._cell_rectangle(ix, iy, dx, dy))

            elif method == "boundary":
                poly = self._trace_boundary(cells, dx, dy)
                if poly is not None:
                    polygons.append(poly)

            else:
                raise ValueError(
                    f"Unknown method '{method}'. Use 'boundary' or 'per_cell'."
                )

        if print_polygons:
            for i, poly in enumerate(polygons):
                print(f"Polygon {i}: {poly}")

        return polygons

    @staticmethod
    def _cell_rectangle(
        ix: int,
        iy: int,
        dx: float,
        dy: float,
    ) -> list[tuple[float, float]]:
        """
        Return the four CCW vertices of grid cell (ix, iy) in physical
        coordinates. Origin is at the bottom-left of the domain.
        """
        x0, x1 = ix * dx,       (ix + 1) * dx
        y0, y1 = iy * dy,       (iy + 1) * dy
        return [(x0, y0), (x1, y0), (x1, y1), (x0, y1)]

    @staticmethod
    def _trace_boundary(
        cells: np.ndarray,
        dx: float,
        dy: float,
    ) -> list[tuple[float, float]] | None:
        """
        Trace the exact outer boundary of one connected solid component.

        Returns a list of (x, y) corner vertices in CCW order with
        collinear vertices removed, or None if tracing fails.

        The boundary is guaranteed to coincide exactly with cell edges,
        so no fluid cell is ever enclosed by the returned polygon.
        """
        from collections import defaultdict

        cell_set = set(map(tuple, cells))

        # ── Step 1: collect exposed cell edges ────────────────────────
        # An edge is exposed when the cell on the other side is not solid.
        # Edges are directed CCW around the solid region:
        #   bottom edge: left→right    top edge: right→left
        #   left edge:   top→bottom    right edge: bottom→top
        boundary_edges = []
        for iy, ix in cells:
            x0, x1 = ix * dx,       (ix + 1) * dx
            y0, y1 = iy * dy,       (iy + 1) * dy

            if (iy - 1, ix) not in cell_set:   # bottom exposed
                boundary_edges.append(((x0, y0), (x1, y0)))
            if (iy + 1, ix) not in cell_set:   # top exposed
                boundary_edges.append(((x1, y1), (x0, y1)))
            if (iy, ix - 1) not in cell_set:   # left exposed
                boundary_edges.append(((x0, y1), (x0, y0)))
            if (iy, ix + 1) not in cell_set:   # right exposed
                boundary_edges.append(((x1, y0), (x1, y1)))

        if not boundary_edges:
            return None

        # ── Step 2: build vertex adjacency map ────────────────────────
        # Round to avoid float equality drift between independently
        # computed edge endpoints that should be the same physical point.
        def snap(pt):
            return (round(pt[0], 10), round(pt[1], 10))

        adj = defaultdict(list)
        for p0, p1 in boundary_edges:
            s0, s1 = snap(p0), snap(p1)
            adj[s0].append(s1)
            adj[s1].append(s0)

        # ── Step 3: walk the polygon ───────────────────────────────────
        # Every vertex in a valid rectilinear boundary has degree exactly 2.
        # We walk by always choosing the unvisited neighbour until we
        # return to the start vertex.
        start   = next(iter(adj))
        polygon = [start]
        prev    = None
        current = start

        for _ in range(len(adj) + 1):
            nxt = next((nb for nb in adj[current] if nb != prev), None)
            if nxt is None or nxt == start:
                break
            polygon.append(nxt)
            prev, current = current, nxt

        if len(polygon) < 3:
            return None

        # ── Step 4: remove collinear vertices ─────────────────────────
        # A vertex is collinear if it lies on the straight line between
        # its two neighbours (cross product of the two edge vectors is 0).
        def are_collinear(a, b, c):
            return abs(
                (b[0] - a[0]) * (c[1] - a[1]) -
                (b[1] - a[1]) * (c[0] - a[0])
            ) < 1e-9

        n = len(polygon)
        polygon = [
            polygon[i]
            for i in range(n)
            if not are_collinear(
                polygon[(i - 1) % n],
                polygon[i],
                polygon[(i + 1) % n],
            )
        ]

        if len(polygon) < 3:
            return None

        # ── Step 5: ensure CCW orientation ────────────────────────────
        def signed_area(pts):
            n_ = len(pts)
            return sum(
                pts[i][0] * pts[(i + 1) % n_][1] -
                pts[(i + 1) % n_][0] * pts[i][1]
                for i in range(n_)
            ) / 2.0

        if signed_area(polygon) < 0:
            polygon = polygon[::-1]

        return polygon
    
    def run_diagnostic(self):
        # Compare CNN output and binarized occupancy grid
        z = torch.randn(1, self.latent_dim, device=self.device)

        self.cnn.eval()
        with torch.no_grad():
            soft_grid = self.cnn(z)

        occupancy_np = soft_grid.squeeze().cpu().numpy()

        print("=== Raw CNN output stats ===")
        print(f"  shape:  {occupancy_np.shape}")
        print(f"  min:    {occupancy_np.min():.4f}")
        print(f"  max:    {occupancy_np.max():.4f}")
        print(f"  mean:   {occupancy_np.mean():.4f}")

        binary = (occupancy_np >= self.threshold)
        print(f"\n=== After threshold ({self.threshold}) ===")
        print(f"  solid density: {binary.mean():.4f}  (valid range: {self.filter.min_density}–{self.filter.max_density})")
        print(f"  density_ok:    {self.filter.min_density <= binary.mean() <= self.filter.max_density}")

        valid, report = self.filter.is_valid(occupancy_np)
        print(f"\n=== Filter report ===")
        for k, v in report.items():
            print(f"  {k}: {v}")

        # Visualize the occupancy grid
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].imshow(occupancy_np, origin='lower', vmin=0, vmax=1, cmap='gray_r')
        axes[0].set_title(f"Soft occupancy (mean={occupancy_np.mean():.3f})")
        axes[0].set_xlabel("x (grid_nx)")
        axes[0].set_ylabel("y (grid_ny)")
        plt.colorbar(axes[0].images[0], ax=axes[0])

        axes[1].imshow(binary, origin='lower', cmap='gray_r')
        axes[1].set_title(f"Binary (threshold={self.threshold}, density={binary.mean():.3f})")
        axes[1].set_xlabel("x (grid_nx)")
        axes[1].set_ylabel("y (grid_ny)")
        plt.tight_layout()
        plt.show()

    def plot_training_curves(self):
        '''
         Visualize training history logged during pretraining.
         Plots loss curves and filter pass rate over training steps.
        '''
        # Plot training curves
        fig, axes = plt.subplots(2, 1, figsize=(12, 8))

        steps = [h["step"] for h in self.history]
        axes[0].plot(steps, [h["total_loss"] for h in self.history], label="total", linewidth=2)
        for key in ["flow_loss", "density_loss", "diversity_loss", "fragmentation_loss", "tortuosity_loss"]:
            axes[0].plot(steps, [h[key] for h in self.history], label=key.replace("_loss", ""), alpha=0.7)
        axes[0].set_xlabel("Step")
        axes[0].set_ylabel("Loss (unweighted)")
        axes[0].legend(fontsize=8)
        axes[0].set_title("Training loss curves")

        pass_rates = [h["filter_pass_rate"] for h in self.history if not np.isnan(h["filter_pass_rate"])]
        pass_steps = [h["step"] for h in self.history if not np.isnan(h["filter_pass_rate"])]
        axes[1].plot(pass_steps, pass_rates, marker="o", markersize=3)
        axes[1].set_xlabel("Step")
        axes[1].set_ylabel("Filter pass rate")
        axes[1].set_ylim(0, 1)
        axes[1].set_title("Filter pass rate during training")

        plt.tight_layout()
        plt.show()

    def plot_mesh_diagnostic(
        self,
        hx: "HeatExchanger",
        show_polygon_labels: bool = True,
    ) -> None:
        """
        Side-by-side diagnostic plot comparing the binary occupancy grid
        to the GMSH mesh for the same heat exchanger design.
 
        Parameters
        ----------
        hx : HeatExchanger
            A HeatExchanger object whose mesh has already been generated
            (hx.mesh_exists must be True).
        show_polygon_labels : bool
            If True, annotate each polygon rectangle with its index.
        """
        import matplotlib.pyplot as plt
        import matplotlib.patches as mpatches
        from matplotlib.collections import LineCollection
        import matplotlib.colors as mcolors
 
        gc = hx.geometry_config
        if gc is None:
            print("[diagnostic] No GeometryConfig attached to this HeatExchanger.")
            return
        if not hx.mesh_exists:
            print("[diagnostic] Mesh file not found — call hx.generate_mesh() first.")
            return
 
        occupancy_np = np.array(gc.occupancy_grid, dtype=np.float32)
        binary       = (occupancy_np >= self.threshold).astype(np.float32)
 
        # ── Figure setup ──────────────────────────────────────────────
        fig, axes = plt.subplots(
            1, 2,
            figsize=(14, 5),
            gridspec_kw={"wspace": 0.12},
        )
        ax_grid, ax_mesh = axes
 
        L = gc.domain_length
        H = gc.domain_height
 
        # ── Left: binary occupancy + polygon overlays ─────────────────
        # imshow: extent maps pixel edges to physical coords.
        # origin='lower' so row 0 (iy=0) is at the bottom of the plot.
        # y runs from -H/2 to +H/2 to match the GMSH coordinate system.
        ax_grid.imshow(
            binary,
            origin="lower",
            extent=[0, L, -H / 2, H / 2],
            cmap="gray_r",        # solid=black, fluid=white
            vmin=0, vmax=1,
            interpolation="nearest",
            aspect="auto",
        )
 
        # Overlay obstacle polygons
        poly_colors = list(mcolors.TABLEAU_COLORS.values())
        polygons    = gc.obstacle_polygons or []
 
        for i, poly in enumerate(polygons):
            color  = poly_colors[i % len(poly_colors)]
            xs     = [p[0] for p in poly] + [poly[0][0]]
            ys_raw = [p[1] for p in poly] + [poly[0][1]]
            # Polygon y coordinates are in [0, H]; shift to [-H/2, +H/2]
            ys     = [y - H / 2 for y in ys_raw]
            ax_grid.plot(xs, ys, color=color, linewidth=1.8, linestyle="--", zorder=3)
 
            if show_polygon_labels:
                cx = sum(p[0] for p in poly) / len(poly)
                cy = sum(p[1] for p in poly) / len(poly) - H / 2
                ax_grid.text(
                    cx, cy, str(i),
                    color=color, fontsize=7, fontweight="bold",
                    ha="center", va="center", zorder=4,
                )
 
        # Inlet / outlet marker lines
        for x_val, label_txt in [(0, "Inlet"), (L, "Outlet")]:
            ax_grid.axvline(x_val, color="red", linewidth=1.2,
                            linestyle=":", alpha=0.8, zorder=5)
            ax_grid.text(
                x_val + (0.01 * L if x_val == 0 else -0.01 * L),
                H / 2 * 0.85,
                label_txt,
                color="red", fontsize=7,
                ha="left" if x_val == 0 else "right",
                va="top",
            )
 
        ax_grid.set_xlim(0, L)
        ax_grid.set_ylim(-H / 2, H / 2)
        ax_grid.set_xlabel("x [m]")
        ax_grid.set_ylabel("y [m]")
        ax_grid.set_title(
            f"Binary occupancy + GMSH polygons\n"
            f"Grid {gc.grid_nx}x{gc.grid_ny}  |  "
            f"Solid density: {binary.mean():.3f}  |  "
            f"{len(polygons)} polygon(s)"
        )
 
        # Legend for polygon colours
        if polygons and show_polygon_labels:
            handles = [
                mpatches.Patch(
                    facecolor="none",
                    edgecolor=poly_colors[i % len(poly_colors)],
                    linestyle="--",
                    linewidth=1.5,
                    label=f"Poly {i}",
                )
                for i in range(len(polygons))
            ]
            ax_grid.legend(
                handles=handles,
                fontsize=6,
                loc="upper right",
                framealpha=0.6,
                ncol=max(1, len(polygons) // 8),
            )
 
        # ── Right: GMSH mesh edges ─────────────────────────────────────
        # Read the mesh with gmsh, extract edges per physical group, and
        # draw them colour-coded.  We use meshio for a lightweight read
        # that doesn't require an active gmsh session.
        try:
            import meshio
            self._plot_mesh_panel(ax_mesh, hx.mesh_filename, L, H)
        except ImportError:
            # Fall back to a plain PyVista point-cloud view if meshio is absent
            try:
                import pyvista as pv
                m = pv.read(hx.mesh_filename)
                pts = np.array(m.points)
                ax_mesh.scatter(
                    pts[:, 0], pts[:, 1],
                    s=0.3, c="black", linewidths=0,
                )
                ax_mesh.set_title("Mesh nodes (install meshio for edge plot)")
            except Exception as e:
                ax_mesh.text(
                    0.5, 0.5,
                    f"Could not load mesh:\n{e}",
                    transform=ax_mesh.transAxes,
                    ha="center", va="center", fontsize=9,
                )
 
        ax_mesh.set_xlim(0, L)
        ax_mesh.set_ylim(-H / 2, H / 2)
        ax_mesh.set_xlabel("x [m]")
        ax_mesh.set_ylabel("y [m]")
 
        fig.suptitle(f"Mesh diagnostic — {hx.id}", fontsize=10, y=1.01)
        plt.tight_layout()
        plt.show()
 
    def _plot_mesh_panel(
        self,
        ax,
        mesh_filename: str,
        domain_length: float,
        domain_height: float,
    ) -> None:
        """
        Draw mesh edges colour-coded by physical group onto ax using meshio.
 
        Physical groups and their colours
        ----------------------------------
        Fluid domain (2D triangles/quads)  : light blue fill, no edge
        Inlet (1D lines)                   : green
        Outlet (1D lines)                  : orange
        Top / Bottom (1D lines)            : grey
        Wall  (obstacle surfaces, 1D)      : dark red
 
        The y-coordinates in the .msh file run from 0 to domain_height
        (GMSH convention used by HeatExchanger.generate_mesh).
        """
        import meshio
        import matplotlib.patches as mpatches
        from matplotlib.collections import LineCollection
 
        H = domain_height
        mesh = meshio.read(mesh_filename)
 
        # All node coordinates — shift y to [-H/2, +H/2]
        pts = mesh.points[:, :2].copy()
 
        # Colour map for named physical groups
        group_styles = {
            "Fluid":  {"color": "#aad4f5", "lw": 0.0,  "zorder": 1},
            "Inlet":  {"color": "#2ca02c", "lw": 1.5,  "zorder": 4},
            "Outlet": {"color": "#ff7f0e", "lw": 1.5,  "zorder": 4},
            "Top":    {"color": "#7f7f7f", "lw": 0.8,  "zorder": 3},
            "Bottom": {"color": "#7f7f7f", "lw": 0.8,  "zorder": 3},
            "Wall":   {"color": "#8B0000", "lw": 1.0,  "zorder": 3},
        }
 
        ax.set_facecolor("white")
 
        legend_handles = []
        drawn_groups   = set()
 
        for cell_block in mesh.cells:
            # Recover the physical group name for this block
            group_name = None
            for name, tag_array in mesh.cell_sets.items():
                # meshio cell_sets: dict[name, list-of-arrays-per-block]
                # Each entry is a list with one array per cell block;
                # the array contains indices of cells in that block that
                # belong to the group.
                for block_idx, indices in enumerate(tag_array):
                    if block_idx == mesh.cells.index(cell_block) and len(indices) > 0:
                        group_name = name
                        break
                if group_name:
                    break
 
            style = group_styles.get(group_name, {"color": "#888888", "lw": 0.5, "zorder": 2})
 
            if cell_block.type in ("line", "line3"):
                # 1D boundary edges
                segs = []
                for edge in cell_block.data:
                    p0 = pts[edge[0]]
                    p1 = pts[edge[-1]]   # edge[-1] works for both line and line3
                    segs.append([p0, p1])
                if segs:
                    lc = LineCollection(
                        segs,
                        colors=style["color"],
                        linewidths=style["lw"],
                        zorder=style["zorder"],
                    )
                    ax.add_collection(lc)
 
            elif cell_block.type in ("triangle", "triangle6", "quad", "quad8", "quad9"):
                # 2D fluid elements — draw filled patches lightly, then edges
                edge_segs = []
                for elem in cell_block.data:
                    # Corner nodes only (skip mid-side nodes for display)
                    n_corners = 3 if "triangle" in cell_block.type else 4
                    corners   = elem[:n_corners]
                    poly_pts  = pts[corners]
                    # Close the polygon
                    edge_segs.append(
                        np.vstack([poly_pts, poly_pts[0]])
                    )
 
                # Thin grey edges for all elements
                segs = [s for s in edge_segs]
                lc = LineCollection(
                    [s for s in edge_segs],
                    colors="#cccccc",
                    linewidths=0.2,
                    zorder=style["zorder"],
                )
                ax.add_collection(lc)
 
            if group_name and group_name not in drawn_groups:
                s = group_styles.get(group_name, {"color": "#888888"})
                legend_handles.append(
                    mpatches.Patch(color=s["color"], label=group_name)
                )
                drawn_groups.add(group_name)
 
        ax.autoscale_view()
        ax.set_title(
            f"GMSH mesh\n"
            f"{sum(len(b.data) for b in mesh.cells if b.type in ('triangle','triangle6','quad','quad8','quad9'))} elements"
        )
        if legend_handles:
            ax.legend(
                handles=legend_handles,
                fontsize=7,
                loc="upper right",
                framealpha=0.6,
            )


In [ ]:
# Initialize the HeatExchangerGenerator with desired parameters
domain_length = 2.0
domain_height = 1.0
grid_nx = 200
grid_ny = 100

hx_gen = HeatExchangerGenerator(
    grid_nx=grid_nx,
    grid_ny=grid_ny,
    domain_length=domain_length,
    domain_height=domain_height,
    latent_dim=32,
    threshold=0.3,
    device=device,
    min_density=0.15,
    max_density=0.80,
    config_directory="configs",
    mesh_directory="meshes",
    mesh_params={
        "mesh_algorithm": 8,
        "mesh_recombine": 1,
        "mesh_element_order": 2
    }
)

# Pretrain the CNN to produce filter-valid designs using only the differentiable FilterLoss — no simulation required.
history = hx_gen.pretrain_on_filter(n_steps=2000, batch_size=16, lr=1e-3, log_every=100)
hx_gen.plot_training_curves()

# Load Pretrained weights if available
# hx_gen.load_weights(f"problems/{grid_nx}x{grid_ny}-generator_weights.pth")

# Run the diagnostic to visualize the CNN output and filter checks after pretraining.
# hx_gen.run_diagnostic()

# Save the trained weights
trained = False
if trained:
    # Avoid overwriting existing weights if already trained and performance not improved
    hx_gen.save_weights("problems/generator_weights.pth")